In [34]:
#%pip install pypdf

In [35]:
import os
import numpy as np
import time
from pypdf import PdfReader
from common import client, ask

In [36]:
PDF="오로라테크_사내규정집_v7.0.pdf"
# EMBED_MODEL = "gemini-embedding-001"
EMBED_MODEL = "gemini-embedding-2"

# 04예제는 저장될 문서가 문자열 리스트이고, 지금 예제는 pdf

# 1. pdf -> 조각
# size : 몇자씩 조각낼건지
# overlap : 몇자씩 겹치게 할건지 
def load_chunks(path, size=500, overlap=100):
    chunks = []
    for page_no, page in enumerate(PdfReader(path).pages, start=1):
        # page.extract_text() : pdf 페이지 내용
        # .replace("\n", " ") : 엔터를 스페이스로 변경
        # .strip() : 시작 앞 공백과 마지막 글자 뒤 공백을 제거
        text = (page.extract_text() or "").replace("\n", " ").strip()
        # size개수만큼 문자열을 가져온고, 다음 작업은 100개가 겹치게 시작점을 지정
        for start in range(0, len(text), size - overlap):
            # size개수만큼 문자열을 가져옴
            piece = text[start: start+size]
            if len(piece) > 50: # 작은 조각(50자이하)은 버림
                # 답변할 때 규정집 어디에 있는지 알리기 위해 페이지 번호도 함께 
                chunks.append({"page" : page_no, "text" : piece})
    return chunks    

In [37]:
chunks = load_chunks(PDF)

In [38]:
# 04예제와 다른점은 04 예제는 임베딩할 문자열이 몇개 안되서 통째로 임베딩을 함
# 이 예제는 문서를 불러와서 임베딩 하기 때문에 임베딩할 문자열이 많을 수 있음
# batch를 이용하여 임베딩할 문자열 개수를 조절하여 나누어서 임베딩을 함
def embed(texts, batch=50):
    vecs = []
    if EMBED_MODEL == "gemini-embedding-001":
        for i in range(0, len(texts), batch):
            r = client.models.embed_content(model=EMBED_MODEL, contents=texts)
            vecs += [e.values for e in r.embeddings]
            time.sleep(0.5)
    elif EMBED_MODEL == "gemini-embedding-2":
        count = 0
        for text in texts:
            r = client.models.embed_content(model=EMBED_MODEL, contents=text)
            vecs.append(r.embeddings[0].values)
            count+=1
            print(f"{count}/{len(texts)} 완료",end=" ") # 확인용
            time.sleep(0.5)
    return np.array(vecs)

In [39]:
VECS_FILE = "pdf_vecs.npy"
if os.path.exists(VECS_FILE):
    vecs = np.load(VECS_FILE)
else:
    vecs = embed([c.get("text","") for c in chunks])
    np.save(VECS_FILE, vecs)
print(f"조각 {len(chunks)}개 준비 완료")

조각 76개 준비 완료


In [43]:
def search(question, k=3):
    q = embed([question])[0]
    # 코사인 유사도를 계산. 저장된 문서 중 질문과 가장 가까운 문서를 찾기 위해 
    sims = vecs @ q / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(q))
    # 유사도가 높은 순으로 정렬 후 k개를 선택
    top = np.argsort(sims)[::-1][:k]          # 2. 검색: 유사도 높은 순
    return [(chunks[i], round(float(sims[i]), 3)) for i in top]

In [44]:
def answer(question):
    found = search(question)
    context = "\n".join(f"[{i+1}] {t}" for i, (t, _) in enumerate(found))
    r = ask(f"""아래 문서에 있는 내용으로만 답하라. 문서에 없으면 '자료에 없습니다'라고 답한다.
    답 끝에 참고한 페이지를 (p.번호) 형식으로 적는다.
    
    {context}
    
    질문: {question}""", temperature=0)        # 3. 답변
    return r.text

In [ ]:
while True:
    q = input("질문 : ")
    if q == "그만":
        break
    print(answer(q), "\n")

질문 :  신입은 휴가가 몇일이야?


  (gemini-3.1-flash-lite 응답)
계속 근로 기간이 1년 미만인 직원은 1개월 개근 시 1일의 유급휴가를 부여받으며, 최대 11일까지 사용할 수 있습니다. (p.11) 



질문 :  경력직은 휴가가 몇일이야?


  (gemini-3.1-flash-lite 응답)
제공된 자료에는 '경력직'의 휴가 일수에 대해 별도로 명시된 내용이 없습니다. (p.11, 12) 



질문 :  휴가가 몇일이야?


  (gemini-3.1-flash-lite 응답)
질문하신 휴가의 종류에 따른 일수는 다음과 같습니다.

*   **연차휴가:** 1년간 80% 이상 출근 시 15일이 부여되며, 3년 이상 계속 근로한 경우 가산휴가를 포함하여 최대 25일까지 부여됩니다. 1년 미만 또는 80% 미만 출근 시에는 1개월 개근 시 1일(최대 11일)이 부여됩니다. (p.11)
*   **병가:** 연간 60일 이내로 사용할 수 있습니다. (p.12)
*   **리프레시 휴가:** 근속 3년이 되는 시점마다 10일이 부여됩니다. (p.12)

(p.11, p.12) 



질문 :  휴가 신청하려면 어떻게 해야되?


  (gemini-3.1-flash-lite 응답)
연차휴가는 사용 예정일 3근무일 전까지 HR포털로 신청해야 합니다. 다만, 반반차는 전일까지 신청할 수 있습니다. 경조휴가의 경우 청첩장, 가족관계증명서, 사망진단서 등 증빙 서류를 HR포털에 첨부하여 신청해야 합니다. (p.11) 



질문 :  출근과 퇴근은 몇시야?


  (gemini-3.1-flash-lite 응답)
소정근로시간은 1일 8시간이며, 선택적 근로시간제 적용 시 출근 가능 시간은 07:00부터 10:00까지입니다. 퇴근 시간에 대한 구체적인 명시는 자료에 없습니다. (p.9) 

